# Build Baselines - Constant, Linear, XGB, LGB, CatBoost

We build baseline models and save each one's CV OOF and test PRED to disk for the stacking, hill-climbing, and pseudo-label notebooks. All models run on CPU and are scored by AUC. SVR is skipped (O(n^2) on 439k rows). Each tree model uses **multi-seed averaging**; LightGBM is **Optuna-tuned**; and three **per-fold target encodings** (`Driver`, `Driver x Compound`, `Race x Compound`) are computed inside every fold.

In [ ]:
VER = 1

## Load Data

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

train = pd.read_parquet('data/train_features.parquet')
print('Train shape:', train.shape)
train.head(1)

In [ ]:
test = pd.read_parquet('data/test_features.parquet')
print('Test shape:', test.shape)
test.head(1)

In [ ]:
TARGET = 'PitNextLap'

# Auto-detect features: everything except id, target, and the raw categoricals used only for TE.
# 'Race' stays in the tree feature set (handled natively); the linear model drops it.
DROP = ['id', TARGET, 'Driver', 'Compound']
FEATURES_TREE   = [c for c in train.columns if c not in DROP]
FEATURES_LINEAR = [c for c in FEATURES_TREE if c != 'Race']
TE_COLS = ['driver_pit_rate', 'driver_compound_pit_rate', 'race_compound_pit_rate']
print(f'tree features:   {len(FEATURES_TREE)}  (+{len(TE_COLS)} per-fold TEs)')
print(f'linear features: {len(FEATURES_LINEAR)}  (+{len(TE_COLS)} per-fold TEs)')

## Cross-Validation Setup

10-fold `StratifiedGroupKFold` grouped by `(Race, Year)`. `fold_rates` builds the three target-encoding tables from the training fold only; `add_te` maps them onto any split.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 10
RANDOM_STATE = 42
y = train[TARGET].astype(int).values
groups = train.groupby(['Race', 'Year']).ngroup().values
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def fold_rates(tr):
    s = pd.Series(y[tr])
    gmean = s.mean()
    drv = s.groupby(train['Driver'].iloc[tr].values).mean()
    dc  = s.groupby([train['Driver'].iloc[tr].values, train['Compound'].iloc[tr].values]).mean()
    rc  = s.groupby([train['Race'].iloc[tr].values, train['Compound'].iloc[tr].values]).mean()
    return gmean, drv, dc, rc

def add_te(base_df, src_df, rates):
    gmean, drv, dc, rc = rates
    out = base_df.copy()
    out['driver_pit_rate'] = src_df['Driver'].map(drv).fillna(gmean).values
    out['driver_compound_pit_rate'] = dc.reindex(
        pd.MultiIndex.from_arrays([src_df['Driver'], src_df['Compound']])).fillna(gmean).values
    out['race_compound_pit_rate'] = rc.reindex(
        pd.MultiIndex.from_arrays([src_df['Race'], src_df['Compound']])).fillna(gmean).values
    return out

## Constant Model

Predicts the mean target for everyone. With AUC, a constant prediction is tied at 0.5 — the floor every other model must beat.

In [ ]:
p = train[TARGET].mean()
print(f'Train pit rate = {p:.5f}')
print('Constant Model Baseline:')
print(' => OOF AUC = 0.50000 (all predictions tied)')

## Logistic Regression (LR)

Linear baseline on standardized numeric features + the three per-fold target encodings (`Race` dropped). NaNs are filled with train-fold means before scaling.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

oof_linear  = np.zeros(len(train))
pred_linear = np.zeros((N_SPLITS, len(test)))

for fold, (tr, va) in enumerate(cv.split(train, y, groups)):
    rates = fold_rates(tr)
    X_tr = add_te(train[FEATURES_LINEAR].iloc[tr], train.iloc[tr], rates)
    X_va = add_te(train[FEATURES_LINEAR].iloc[va], train.iloc[va], rates)
    X_te = add_te(test[FEATURES_LINEAR],           test,           rates)

    fill = X_tr.mean()
    X_tr, X_va, X_te = X_tr.fillna(fill), X_va.fillna(fill), X_te.fillna(fill)

    scaler = StandardScaler()
    Xs_tr = scaler.fit_transform(X_tr)
    Xs_va = scaler.transform(X_va)
    Xs_te = scaler.transform(X_te)

    model = LogisticRegression(C=1.0, max_iter=2000)
    model.fit(Xs_tr, y[tr])
    oof_linear[va]    = model.predict_proba(Xs_va)[:, 1]
    pred_linear[fold] = model.predict_proba(Xs_te)[:, 1]
    print(f'Fold {fold+1} AUC: {roc_auc_score(y[va], oof_linear[va]):.5f}')

print('-' * 40)
print(f'Linear Model, CV OOF AUC: {roc_auc_score(y, oof_linear):.5f}')

In [ ]:
importance = np.abs(model.coef_[0])
names = FEATURES_LINEAR + TE_COLS
imp_df = pd.DataFrame({'feature': names, 'importance': importance}).sort_values('importance', ascending=False)
plt.figure(figsize=(6, 9))
plt.barh(imp_df['feature'].head(30), imp_df['importance'].head(30))
plt.gca().invert_yaxis()
plt.title('Logistic Regression |coef| (top 30, last fold)')
plt.tight_layout()
plt.show()

In [ ]:
np.save(f'data/train_oof_linear_v{VER}.npy', oof_linear)
np.save(f'data/test_pred_linear_v{VER}.npy', pred_linear)

## SVR — skipped

Polynomial-kernel SVR is O(n^2) in the kernel matrix; at 439k rows on CPU it is hours per fold. Non-linearity is covered by the three tree models below.

## Tree Feature Matrix

Shared by XGBoost, LightGBM, and CatBoost. `Race` is set to a pandas category.

In [ ]:
X      = train[FEATURES_TREE].copy()
X_test = test[FEATURES_TREE].copy()
X['Race']      = X['Race'].astype('category')
X_test['Race'] = pd.Categorical(X_test['Race'], categories=X['Race'].cat.categories)

## XGBoost Model

Non-linear baseline with native categorical support and **3-seed averaging** per fold.

In [ ]:
from xgboost import XGBClassifier

SEEDS_XGB = [42, 17, 2024]
xgb_params = dict(
    n_estimators=2000, max_depth=6, learning_rate=0.05, min_child_weight=10,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    tree_method='hist', enable_categorical=True, eval_metric='auc',
    early_stopping_rounds=100, verbosity=0,
)

oof_xgb  = np.zeros(len(train))
pred_xgb = np.zeros((N_SPLITS, len(test)))

for fold, (tr, va) in enumerate(cv.split(X, y, groups)):
    rates = fold_rates(tr)
    X_tr = add_te(X.iloc[tr], train.iloc[tr], rates)
    X_va = add_te(X.iloc[va], train.iloc[va], rates)
    X_te = add_te(X_test,      test,           rates)
    fva = np.zeros(len(va)); fte = np.zeros(len(test))
    for seed in SEEDS_XGB:
        m = XGBClassifier(**xgb_params, random_state=seed)
        m.fit(X_tr, y[tr], eval_set=[(X_va, y[va])], verbose=False)
        fva += m.predict_proba(X_va)[:, 1] / len(SEEDS_XGB)
        fte += m.predict_proba(X_te)[:, 1] / len(SEEDS_XGB)
    oof_xgb[va]    = fva
    pred_xgb[fold] = fte
    print(f'Fold {fold+1} AUC: {roc_auc_score(y[va], fva):.5f}')

print('-' * 40)
print(f'XGB Model, CV OOF AUC: {roc_auc_score(y, oof_xgb):.5f}')

In [ ]:
score = m.get_booster().get_score(importance_type='gain')
imp = pd.DataFrame({'feature': list(score.keys()), 'importance': list(score.values())})\
        .sort_values('importance', ascending=False)
plt.figure(figsize=(6, 9))
plt.barh(imp['feature'].head(30), imp['importance'].head(30))
plt.gca().invert_yaxis()
plt.title('XGBoost Feature Importance (gain, top 30)')
plt.tight_layout()
plt.show()

In [ ]:
np.save(f'data/train_oof_xgb_v{VER}.npy', oof_xgb)
np.save(f'data/test_pred_xgb_v{VER}.npy', pred_xgb)

## Hyperparameter Tuning (LightGBM, Optuna)

50 trials on the first fold (fast). The best params feed the LightGBM CV below. Trial progress prints via Optuna's INFO logging.

In [ ]:
import optuna
import lightgbm as lgb

tr0, va0 = next(iter(cv.split(X, y, groups)))
rates0 = fold_rates(tr0)
X0_tr = add_te(X.iloc[tr0], train.iloc[tr0], rates0)
X0_va = add_te(X.iloc[va0], train.iloc[va0], rates0)

def objective(trial):
    params = {
        'n_estimators': 1500,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'bagging_freq': 5, 'random_state': 42, 'verbose': -1,
    }
    m = lgb.LGBMClassifier(**params)
    m.fit(X0_tr, y[tr0], eval_set=[(X0_va, y[va0])],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    return roc_auc_score(y[va0], m.predict_proba(X0_va)[:, 1])

optuna.logging.set_verbosity(optuna.logging.INFO)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=False)
best_params = {**study.best_params, 'n_estimators': 2000, 'bagging_freq': 5, 'random_state': 42, 'verbose': -1}
print(f'best tuning AUC: {study.best_value:.5f}')
print(f'best params: {best_params}')

## LightGBM Model

Optuna-tuned params with **5-seed averaging** per fold.

In [ ]:
SEEDS_LGB = [42, 17, 2024, 1337, 9999]
oof_lgb  = np.zeros(len(train))
pred_lgb = np.zeros((N_SPLITS, len(test)))

for fold, (tr, va) in enumerate(cv.split(X, y, groups)):
    rates = fold_rates(tr)
    X_tr = add_te(X.iloc[tr], train.iloc[tr], rates)
    X_va = add_te(X.iloc[va], train.iloc[va], rates)
    X_te = add_te(X_test,      test,           rates)
    fva = np.zeros(len(va)); fte = np.zeros(len(test))
    for seed in SEEDS_LGB:
        m = lgb.LGBMClassifier(**{**best_params, 'random_state': seed})
        m.fit(X_tr, y[tr], eval_set=[(X_va, y[va])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        fva += m.predict_proba(X_va)[:, 1] / len(SEEDS_LGB)
        fte += m.predict_proba(X_te)[:, 1] / len(SEEDS_LGB)
    oof_lgb[va]    = fva
    pred_lgb[fold] = fte
    print(f'Fold {fold+1} AUC: {roc_auc_score(y[va], fva):.5f}')

print('-' * 40)
print(f'LightGBM, CV OOF AUC: {roc_auc_score(y, oof_lgb):.5f}')

In [ ]:
imp = pd.DataFrame({'feature': X_tr.columns,
                    'importance': m.booster_.feature_importance(importance_type='gain')})\
        .sort_values('importance', ascending=False)
plt.figure(figsize=(6, 9))
plt.barh(imp['feature'].head(30), imp['importance'].head(30))
plt.gca().invert_yaxis()
plt.title('LightGBM Feature Importance (gain, top 30)')
plt.tight_layout()
plt.show()

In [ ]:
np.save(f'data/train_oof_lgb_v{VER}.npy', oof_lgb)
np.save(f'data/test_pred_lgb_v{VER}.npy', pred_lgb)

## CatBoost Model

Ordered target encoding on `Race` gives errors decorrelated from XGB/LGB. Single seed (CatBoost is slow); `Race` cast to string per fold.

In [ ]:
import catboost as cb

cb_params = dict(
    iterations=2000, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
    random_seed=RANDOM_STATE, early_stopping_rounds=100, eval_metric='AUC',
    cat_features=['Race'], verbose=500, use_best_model=True,
)

oof_cb  = np.zeros(len(train))
pred_cb = np.zeros((N_SPLITS, len(test)))

for fold, (tr, va) in enumerate(cv.split(X, y, groups)):
    rates = fold_rates(tr)
    X_tr = add_te(X.iloc[tr], train.iloc[tr], rates)
    X_va = add_te(X.iloc[va], train.iloc[va], rates)
    X_te = add_te(X_test,      test,           rates)
    for D in (X_tr, X_va, X_te):
        D['Race'] = D['Race'].astype(str)
    m = cb.CatBoostClassifier(**cb_params)
    m.fit(X_tr, y[tr], eval_set=(X_va, y[va]))
    oof_cb[va]    = m.predict_proba(X_va)[:, 1]
    pred_cb[fold] = m.predict_proba(X_te)[:, 1]
    print(f'Fold {fold+1} AUC: {roc_auc_score(y[va], oof_cb[va]):.5f}  best_iter={m.best_iteration_}')

print('-' * 40)
print(f'CatBoost, CV OOF AUC: {roc_auc_score(y, oof_cb):.5f}')

In [ ]:
imp = pd.DataFrame({'feature': X_tr.columns, 'importance': m.get_feature_importance()})\
        .sort_values('importance', ascending=False)
plt.figure(figsize=(6, 9))
plt.barh(imp['feature'].head(30), imp['importance'].head(30))
plt.gca().invert_yaxis()
plt.title('CatBoost Feature Importance (top 30)')
plt.tight_layout()
plt.show()

In [ ]:
np.save(f'data/train_oof_cb_v{VER}.npy', oof_cb)
np.save(f'data/test_pred_cb_v{VER}.npy', pred_cb)

## Summary

OOF AUC by model. Saved arrays feed the stacking / hill-climb / pseudo-label notebooks.

In [ ]:
results = {
    'Constant': 0.5,
    'Linear':   roc_auc_score(y, oof_linear),
    'XGBoost':  roc_auc_score(y, oof_xgb),
    'LightGBM': roc_auc_score(y, oof_lgb),
    'CatBoost': roc_auc_score(y, oof_cb),
}
summary = pd.DataFrame({'OOF AUC': results}).sort_values('OOF AUC', ascending=False)
print(summary.to_string())